# Results for Final Evaluation on Test Split

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

# Locate the data from either the notebook directory or the repository root.
RUNS_DIR = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent / "experiments" / "4_active_learning_runs", parent)
     if candidate.name == "4_active_learning_runs"
     and any(candidate.glob("*/config.json"))),
    None,
)
if RUNS_DIR is None:
    raise FileNotFoundError("Cannot locate experiments/4_active_learning_runs.")

STRATEGIES = {
    "RANDOM_SAMPLING": "Random Sampling (Baseline)",
    "COARSE_UNCERTAINTY_MEASURE": "Coarse UM",
    "COARSE_ENTROPY_MEASURE": "Coarse EM",
    "FINE_UNCERTAINTY_MEASURE": "Fine UM",
    "FINE_ENTROPY_MEASURE": "Fine EM",
    "GROUPED_FINE_ENTROPY": "Grouped Fine EM",
}
GROUPS = ["G_high", "G_mid", "G_low"]
METRICS = {
    "test_f1_event": "Mean F1 Event",
    "test_f1_element": "Mean F1 Element",
    "test_edit": "Mean Edit Score",
    "test_f1_class_agnostic": "Mean F1 Class-Agnostic",
}

records = []
for config_path in sorted(RUNS_DIR.glob("*/config.json")):
    config = json.loads(config_path.read_text())
    strategy = config["query_strategy"]
    if strategy not in STRATEGIES:
        raise ValueError(f"Unknown query strategy: {strategy}")
    assert config["active_learning_budget_unit"] == "percent_of_training_frames"
    for round_dir in sorted(config_path.parent.glob("round_*")):
        if not round_dir.is_dir():
            continue
        results = json.loads((round_dir / "test_results.json").read_text())
        round_number = int(round_dir.name.split("_")[-1])
        # Nominal acquisition budgets align the same round across seeds.
        budget_pct = min(
            config["initial_labeled_pool_size"]
            + round_number * config["query_batch_size"],
            config["max_annotation_budget"],
        )
        record = {
            "run": config_path.parent.name,
            "seed": config["seed"],
            "strategy": strategy,
            "round": round_number,
            "budget_pct": budget_pct,
            "test_f1_class_agnostic": results["test_f1_class_agnostic"],
        }
        for group in GROUPS:
            for metric in ("test_f1_event", "test_f1_element", "test_edit"):
                value = results[group][metric]
                # Edit scores are stored as percentages; F1 scores are fractions.
                record[f"{group}_{metric}"] = value / 100 if metric == "test_edit" else value
        records.append(record)

rounds = pd.DataFrame(records)
if rounds.empty:
    raise ValueError("No active learning results found.")
assert not rounds.duplicated(["strategy", "seed", "budget_pct"]).any()
score_columns = [column for column in rounds if column.startswith(("G_", "test_"))]
assert np.isfinite(rounds[score_columns].to_numpy()).all()
assert rounds[score_columns].ge(0).all().all()
assert rounds[score_columns].le(1).all().all()
print(f"Loaded {rounds['run'].nunique()} runs and {len(rounds)} active learning rounds.")

COLORS = {strategy: plt.get_cmap("tab10")(i)
          for i, strategy in enumerate(list(STRATEGIES)[1:])}
COLORS["RANDOM_SAMPLING"] = "black"


def set_equal_y_scale(axes):
    # Center each panel on its own curves and SD bands, but use the same
    # range width so equal vertical distances represent equal score changes.
    bounds = np.array([[ax.dataLim.ymin, ax.dataLim.ymax] for ax in axes])
    widest_range = np.max(bounds[:, 1] - bounds[:, 0])
    padding = max(widest_range * 0.05, 1e-6)
    shared_span = widest_range + 2 * padding
    for ax, (lower, upper) in zip(axes, bounds):
        center = (lower + upper) / 2
        ax.set_ylim(center - shared_span / 2, center + shared_span / 2)
        # Show each panel's different limits explicitly.
        ax.tick_params(axis="y", labelleft=True)


def plot_learning_curves(metric, groups=None):
    panels = GROUPS if groups is None else groups
    fig, axes = plt.subplots(
        1, len(panels), figsize=(8 * len(panels), 7),
        sharex=True, sharey=False, squeeze=False,
    )
    for ax, group in zip(axes[0], panels):
        column = f"{group}_{metric}" if group else metric
        for strategy, label in STRATEGIES.items():
            curve = (rounds.loc[rounds["strategy"] == strategy]
                     .groupby("budget_pct")[column].agg(["mean", "std"]))
            # pandas std uses ddof=1: sample SD across independent seed runs.
            x = curve.index.to_numpy(dtype=float)
            mean = curve["mean"].to_numpy()
            sd = curve["std"].to_numpy()
            ax.plot(x, mean, color=COLORS[strategy], label=label,
                    linestyle="--" if strategy == "RANDOM_SAMPLING" else "-",
                    marker="o", markersize=3)
            ax.fill_between(x, mean - sd, mean + sd,
                            color=COLORS[strategy], alpha=0.15)
        ax.set(xlim=(0, 100),
               xlabel="Nominal annotation budget", title=group or METRICS[metric])
        ax.set_xticks(np.arange(0, 101, 10))
        ax.xaxis.set_major_formatter(PercentFormatter(100))
        ax.grid(alpha=0.25)
    set_equal_y_scale(axes[0])
    axes[0, 0].set_ylabel(METRICS[metric])
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False,
               fontsize=8)
    fig.suptitle(f"{METRICS[metric]} — mean ± 1 SD across seeds")
    fig.tight_layout(rect=(0, 0.12, 1, 0.94))
    plt.show()


## Learning Curves

In [ ]:
plot_learning_curves("test_f1_event")

In [ ]:
plot_learning_curves("test_f1_element")


In [ ]:
plot_learning_curves("test_edit")


In [ ]:
plot_learning_curves("test_f1_class_agnostic", groups=[None])


# AULC

In [ ]:
# Integrate each seed separately, then summarize AULC across seeds.
# Budgets are fractions (0–1); integrate only the observed 10%–100% range.
# No zero-budget score is invented and the area is not divided by range width.
budget_grids = [tuple(curve.sort_values("budget_pct")["budget_pct"])
                for _, curve in rounds.groupby("run")]
assert len(set(budget_grids)) == 1, "AULC comparisons require matching budget grids."
assert len(budget_grids[0]) >= 2, "AULC requires at least two budget points."
aulc_records = []
for (strategy, seed, run), curve in rounds.groupby(["strategy", "seed", "run"]):
    curve = curve.sort_values("budget_pct")
    x = curve["budget_pct"].to_numpy(dtype=float) / 100
    assert np.all(np.diff(x) > 0)
    record = {"strategy": strategy, "seed": seed, "run": run}
    for group in GROUPS:
        y = curve[f"{group}_test_f1_event"].to_numpy()
        # Trapezoidal integration, compatible with NumPy 1.x and 2.x.
        record[group] = np.sum(np.diff(x) * (y[:-1] + y[1:]) / 2)
    aulc_records.append(record)

aulc_per_run = pd.DataFrame(aulc_records)
aulc_summary = aulc_per_run.groupby("strategy")[GROUPS].agg(["mean", "std"])
aulc_table = pd.DataFrame(index=pd.Index(STRATEGIES.values(), name="Query Strategy"))
# Compare mean AULC with the random-sampling mean at the same granularity.
aulc_pct_change = pd.DataFrame(index=list(STRATEGIES), columns=GROUPS, dtype=float)
for group in GROUPS:
    baseline = aulc_summary.loc["RANDOM_SAMPLING", (group, "mean")]
    if not np.isfinite(baseline) or baseline <= 0:
        raise ValueError(f"{group}: a positive baseline AULC is required for % change.")
    means = aulc_summary[(group, "mean")].reindex(STRATEGIES)
    aulc_pct_change[group] = 100 * (means - baseline) / baseline
    aulc_table[group] = [
        f"{aulc_summary.loc[strategy, (group, 'mean')]:.4f} ± "
        f"{aulc_summary.loc[strategy, (group, 'std')]:.4f} "
        f"({aulc_pct_change.loc[strategy, group]:+.2f}%)"
        for strategy in STRATEGIES
    ]
start, end = budget_grids[0][0], budget_grids[0][-1]
display(aulc_table)


## % of Full-Budget Performance

In [ ]:
def plot_full_budget_performance(metric, groups=None):
    # Each strategy uses its own full-budget mean for the same metric.
    # Scale both mean and sample SD by this fixed reference, so each point is
    # mean(score at X%) / mean(score at 100%), rather than a mean of seed ratios.
    panels = GROUPS if groups is None else groups
    fig, axes = plt.subplots(
        1, len(panels), figsize=(6 * len(panels), 5),
        sharex=True, sharey=False, squeeze=False,
    )
    for ax, group in zip(axes[0], panels):
        column = f"{group}_{metric}" if group else metric
        for strategy, label in STRATEGIES.items():
            strategy_rounds = rounds.loc[rounds["strategy"] == strategy]
            full_budget = strategy_rounds.loc[
                np.isclose(strategy_rounds["budget_pct"], 100), column
            ]
            reference = full_budget.mean()
            if not np.isfinite(reference) or reference <= 0:
                raise ValueError(
                    f"{label}: a positive 100%-budget mean is required for {column}."
                )
            curve = strategy_rounds.groupby("budget_pct")[column].agg(["mean", "std"])
            x = curve.index.to_numpy(dtype=float)
            mean_pct = 100 * curve["mean"].to_numpy() / reference
            sd_pct = 100 * curve["std"].to_numpy() / reference
            ax.plot(
                x, mean_pct, color=COLORS[strategy], label=label,
                linestyle="--" if strategy == "RANDOM_SAMPLING" else "-",
                marker="o", markersize=3,
            )
            ax.fill_between(
                x, mean_pct - sd_pct, mean_pct + sd_pct,
                color=COLORS[strategy], alpha=0.15,
            )
        ax.axhline(100, color="gray", linestyle=":", linewidth=1)
        ax.set(xlim=(0, 100), xlabel="Nominal annotation budget",
               title=group or METRICS[metric])
        ax.set_xticks(np.arange(0, 101, 10))
        ax.xaxis.set_major_formatter(PercentFormatter(100))
        ax.yaxis.set_major_formatter(PercentFormatter(100))
        # Allow values and SD bands above 100% to remain visible.
        ax.grid(alpha=0.25)
    set_equal_y_scale(axes[0])
    axes[0, 0].set_ylabel("Full-budget performance achieved (%)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False,
               fontsize=8)
    fig.suptitle(f"{METRICS[metric]} relative to full budget — mean ± 1 SD")
    fig.tight_layout(rect=(0, 0.12, 1, 0.94))
    plt.show()


plot_full_budget_performance("test_f1_event")


In [ ]:
# Normalize class-agnostic F1 by its own full-budget mean for each strategy.
plot_full_budget_performance("test_f1_class_agnostic", groups=[None])
